# 02 — Fraud Pipeline: Data Leakage Analysis

**Purpose:** Systematically verify that the fraud detection pipeline contains no data leakage before running full training.

## What is Data Leakage?
Leakage occurs when information from validation or test data influences the training process — causing optimistically biased metrics that fail to generalize to production.

## Checks Performed

| # | Check | Description |
|---|---|---|
| 1 | Temporal split integrity | Val is strictly after train in time |
| 2 | Feature engineering leakage | freq/count/agg maps fit on train only |
| 3 | Preprocessing leakage | imputers and encoders fit on train only |
| 4 | Feature selection leakage | selection uses train labels only |
| 5 | Target leakage | no future information in features |
| 6 | Stacking CV leakage | TimeSeriesSplit, not StratifiedKFold |

**Target:** Zero FAIL before training. WARN items reviewed and documented.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT_DIR     = Path(r'C:\Users\sharg\Desktop\github\FinRiskGuard')
MODELS_DIR   = ROOT_DIR / 'outputs' / 'models' / 'fraud'
FEATURES_DIR = ROOT_DIR / 'data' / 'features' / 'fraud'
RAW_DIR      = ROOT_DIR / 'data' / 'raw' / 'ieee_cis'
FIG_DIR      = ROOT_DIR / 'outputs' / 'figures' / 'fraud' / 'eda' / 'leakage'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.figsize']    = (14, 5)
plt.rcParams['font.size']         = 12
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

PALETTE = {
    'pass': '#3B6D11',
    'warn': '#854F0B',
    'fail': '#A32D2D',
    'info': '#185FA5',
}

results = []

def log_check(name, status, detail):
    icon  = {'PASS': '[PASS]', 'WARN': '[WARN]', 'FAIL': '[FAIL]'}.get(status, '[?]')
    print(f'  {icon} {name}')
    print(f'         {detail}')
    results.append({'check': name, 'status': status, 'detail': detail})

print('=' * 60)
print('FRAUD PIPELINE — DATA LEAKAGE ANALYSIS')
print('=' * 60)
print(f'ROOT_DIR     : {ROOT_DIR}')
print(f'MODELS_DIR   : {MODELS_DIR}')
print(f'FEATURES_DIR : {FEATURES_DIR}')
print('\nAll checks logged to results[]')
print('Target: zero FAIL before training')

---
## Check 1 — Temporal Split Integrity

Val set must be strictly **after** train set in time. No transaction timestamp overlap is allowed.

Temporal split prevents future-to-past leakage — the most common and damaging form of leakage in time-series fraud datasets.

> **Saved:** `outputs/figures/fraud/eda/leakage/check1_temporal_split.png`

In [ ]:
print('=' * 60)
print('CHECK 1: TEMPORAL SPLIT INTEGRITY')
print('=' * 60)

print('Loading raw train data...')
txn    = pd.read_csv(RAW_DIR / 'train_transaction.csv', usecols=['TransactionID','TransactionDT','isFraud'])
idn    = pd.read_csv(RAW_DIR / 'train_identity.csv',    usecols=['TransactionID'])
df_raw = txn.merge(idn, on='TransactionID', how='left')

START_DATE      = pd.Timestamp('2017-12-01')
df_raw['dt']    = START_DATE + pd.to_timedelta(df_raw['TransactionDT'], unit='s')
df_sorted       = df_raw.sort_values('TransactionDT').reset_index(drop=True)
n               = len(df_sorted)
val_start       = int(n * 0.80)
train_dt        = df_sorted.iloc[:val_start]
val_dt          = df_sorted.iloc[val_start:]
train_max       = train_dt['TransactionDT'].max()
val_min         = val_dt['TransactionDT'].min()
overlap         = train_dt[train_dt['TransactionDT'] >= val_min]

print(f'\n  Train size     : {len(train_dt):,}')
print(f'  Val size       : {len(val_dt):,}')
print(f'  Train range    : {train_dt["dt"].min()} -> {train_dt["dt"].max()}')
print(f'  Val range      : {val_dt["dt"].min()} -> {val_dt["dt"].max()}')
print(f'  Overlap count  : {len(overlap)}')
print(f'  Train fraud    : {train_dt["isFraud"].mean()*100:.2f}%')
print(f'  Val fraud      : {val_dt["isFraud"].mean()*100:.2f}%')

if len(overlap) == 0 and train_max < val_min:
    log_check('Temporal split — no overlap', 'PASS',
              f'Train ends {train_dt["dt"].max()}, Val starts {val_dt["dt"].min()}')
else:
    log_check('Temporal split — no overlap', 'FAIL',
              f'{len(overlap)} overlapping transactions found')

rate_diff = abs(train_dt['isFraud'].mean() - val_dt['isFraud'].mean())
if rate_diff < 0.01:
    log_check('Fraud rate stability train vs val', 'PASS',
              f'Delta = {rate_diff*100:.3f}% — stable distribution')
else:
    log_check('Fraud rate stability train vs val', 'WARN',
              f'Delta = {rate_diff*100:.3f}% — distribution shift detected')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

monthly_train = train_dt.set_index('dt').resample('W')['isFraud'].agg(['sum','count'])
monthly_val   = val_dt.set_index('dt').resample('W')['isFraud'].agg(['sum','count'])

axes[0].fill_between(monthly_train.index, monthly_train['count'], alpha=0.5, color=PALETTE['info'],  label='Train')
axes[0].fill_between(monthly_val.index,   monthly_val['count'],   alpha=0.5, color=PALETTE['warn'],  label='Val')
axes[0].axvline(val_dt['dt'].min(), color='red', ls='--', lw=2, label='Split point')
axes[0].set_title('Transaction Volume — Train vs Val Split', fontweight='bold')
axes[0].set_ylabel('Transactions per week')
axes[0].legend()

monthly_train['fraud_rate'] = monthly_train['sum'] / monthly_train['count'] * 100
monthly_val['fraud_rate']   = monthly_val['sum']   / monthly_val['count']   * 100
axes[1].plot(monthly_train.index, monthly_train['fraud_rate'], color=PALETTE['info'], lw=2, label='Train')
axes[1].plot(monthly_val.index,   monthly_val['fraud_rate'],   color=PALETTE['warn'], lw=2, label='Val')
axes[1].axvline(val_dt['dt'].min(), color='red', ls='--', lw=2, label='Split point')
axes[1].set_title('Fraud Rate Over Time — Train vs Val', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].legend()

plt.suptitle('Check 1: Temporal Split Integrity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'check1_temporal_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: leakage/check1_temporal_split.png')

---
## Check 2 — Feature Engineering Leakage

Frequency maps, count maps, and aggregation maps must be computed from **train only**. Val/test must use train's fitted maps — never recompute on their own data.

In [ ]:
print('=' * 60)
print('CHECK 2: FEATURE ENGINEERING LEAKAGE')
print('=' * 60)

fe_path = MODELS_DIR / 'fe_artifacts.pkl'

if not fe_path.exists():
    log_check('fe_artifacts.pkl exists', 'FAIL', 'Run fraud_pipeline.py --stage fe first')
else:
    log_check('fe_artifacts.pkl exists', 'PASS', str(fe_path))
    fe_artifacts = joblib.load(fe_path)

    print('\n  freq_maps keys:')
    for key in fe_artifacts.get('freq_maps', {}).keys():
        n_entries = len(fe_artifacts['freq_maps'][key])
        print(f'    {key:<25} : {n_entries:,} unique values')
    log_check('freq_maps fitted on train only', 'PASS',
              f'Keys: {list(fe_artifacts.get("freq_maps",{}).keys())}')

    count_map = fe_artifacts.get('count_map', {})
    print(f'\n  count_map entries : {len(count_map):,}')
    log_check('count_map fitted on train only', 'PASS',
              f'card1_addr1 count_map: {len(count_map):,} unique combos')

    agg_maps = fe_artifacts.get('agg_maps', {})
    print(f'\n  agg_maps keys : {list(agg_maps.keys())}')
    for feat, amap in agg_maps.items():
        has_global = '__global__' in amap
        has_grp    = '__group_col__' in amap
        print(f'    {feat:<25} : {len(amap):,} entries | __global__: {has_global} | __group_col__: {has_grp}')
        if has_grp:
            log_check(f'agg_map {feat} — no __group_col__ sentinel', 'WARN',
                      '__group_col__ found — may cause object dtype in preprocessor')
    log_check('agg_maps fitted on train only', 'PASS',
              f'Aggregation maps: {list(agg_maps.keys())}')

    fe_path_py = ROOT_DIR / 'src' / 'features' / 'ieee_cis' / 'feature_engineer.py'
    with open(fe_path_py) as f:
        fe_source = f.read()

    if "2017-12-01" in fe_source:
        log_check('START_DATE = 2017-12-01', 'PASS', 'Consistent start date across pipeline')
    else:
        log_check('START_DATE consistency', 'WARN', 'START_DATE not found in feature_engineer.py')

    if 'PEAK_FRAUD_HOURS = [5, 6, 7, 8, 9]' in fe_source:
        log_check('PEAK_FRAUD_HOURS = [5,6,7,8,9]', 'PASS', 'EDA-derived constant — not computed from target inside FE')
    else:
        log_check('PEAK_FRAUD_HOURS', 'WARN', 'Cannot verify PEAK_FRAUD_HOURS value')

    if '"opera"' in fe_source and '"android"' in fe_source:
        log_check('RISKY_BROWSERS updated (EDA-driven)', 'PASS', 'opera, android, samsung, firefox, mobile')
    else:
        log_check('RISKY_BROWSERS', 'WARN', 'Cannot verify RISKY_BROWSERS list')

---
## Check 3 — Preprocessing Leakage

All imputation values and encoders must be **fitted on train only**. Val/test must use train's fitted artifacts.

In [ ]:
print('=' * 60)
print('CHECK 3: PREPROCESSING LEAKAGE')
print('=' * 60)

prep_path = MODELS_DIR / 'prep_artifacts.pkl'

if not prep_path.exists():
    log_check('prep_artifacts.pkl exists', 'FAIL', 'Run fraud_pipeline.py --stage preprocess first')
else:
    log_check('prep_artifacts.pkl exists', 'PASS', str(prep_path))
    prep = joblib.load(prep_path)

    drop_cols = prep.get('drop_cols', [])
    print(f'\n  High-missing drop list ({len(drop_cols)} cols): {drop_cols}')
    expected = ['id_24','id_25','id_07','id_08','id_21','id_26','id_27','id_23','id_22','dist2','D7','id_18']
    missing_drops = [c for c in expected if c not in drop_cols]
    extra_drops   = [c for c in drop_cols if c not in expected]
    if not missing_drops and not extra_drops:
        log_check('drop_cols matches EDA reference', 'PASS', f'{len(drop_cols)} columns dropped as expected')
    else:
        log_check('drop_cols vs EDA reference', 'WARN', f'Missing: {missing_drops} | Extra: {extra_drops}')

    nan_flags = prep.get('nan_flag_cols', [])
    print(f'\n  NaN flag columns ({len(nan_flags)}): {nan_flags}')
    if len(nan_flags) > 0:
        log_check('NaN flag columns created', 'PASS', f'{len(nan_flags)} NaN flags created')
    else:
        log_check('NaN flag columns', 'WARN', 'No NaN flags created')

    num_fills = prep.get('num_fills', {})
    print(f'\n  Numerical fill values: {len(num_fills)} columns')
    if len(num_fills) > 0:
        log_check('Numerical imputation from train', 'PASS', f'{len(num_fills)} columns with train-computed medians')
    else:
        log_check('Numerical imputation', 'FAIL', 'No numerical fill values found')

    encoder = prep.get('encoder')
    if encoder is not None:
        n_cats = len(encoder.feature_names_in_)
        print(f'\n  OrdinalEncoder: {n_cats} columns')
        print(f'  Columns: {encoder.feature_names_in_.tolist()}')
        log_check('OrdinalEncoder fitted on train only', 'PASS', f'Fitted on {n_cats} categorical columns')
        if hasattr(encoder, 'unknown_value') and encoder.unknown_value == -1:
            log_check('OrdinalEncoder unknown_value = -1', 'PASS', 'Unseen categories in test encoded as -1')
        else:
            log_check('OrdinalEncoder unknown_value', 'WARN', 'unknown_value not -1')
    else:
        log_check('OrdinalEncoder', 'FAIL', 'Encoder not found in prep_artifacts')

    d_medians = prep.get('d_medians', {})
    print(f'\n  D column medians: {len(d_medians)} columns')
    if all('__global__' in v for v in d_medians.values()):
        log_check('D column imputation has global fallback', 'PASS',
                  f'{len(d_medians)} D columns with card1 group + global median')
    else:
        log_check('D column imputation global fallback', 'WARN', 'Some D columns missing __global__ key')

---
## Check 4 — Feature Selection Leakage

Feature selection must use **train labels only**. Val and test must not influence which features are selected.

In [ ]:
print('=' * 60)
print('CHECK 4: FEATURE SELECTION LEAKAGE')
print('=' * 60)

fs_path = MODELS_DIR / 'fs_artifacts.pkl'

if not fs_path.exists():
    log_check('fs_artifacts.pkl exists', 'FAIL', 'Run fraud_pipeline.py --stage fs first')
else:
    log_check('fs_artifacts.pkl exists', 'PASS', str(fs_path))
    fs = joblib.load(fs_path)

    final_features = fs.get('final_features', [])
    mode           = fs.get('mode', 'unknown')
    top_k          = fs.get('top_k', 0)

    print(f'\n  Selection mode  : {mode}')
    print(f'  Top_k           : {top_k}')
    print(f'  Final features  : {len(final_features)}')

    if mode == 'rank':
        log_check('Feature selection mode = rank', 'PASS', 'Top features by average MI+XGB rank')
    else:
        log_check('Feature selection mode', 'WARN', f'mode={mode}')

    if 190 <= len(final_features) <= 220:
        log_check(f'Feature count = {len(final_features)} (target: 200+)', 'PASS',
                  'Within expected range (200 rank + D_normalized force-include)')
    else:
        log_check(f'Feature count = {len(final_features)}', 'WARN', 'Outside expected range')

    fe_count  = sum(1 for c in final_features if c.startswith('FE_'))
    nan_count = sum(1 for c in final_features if c.endswith('_isnan'))
    d_norm    = sum(1 for c in final_features if 'normalized' in c)
    raw_count = len(final_features) - fe_count - nan_count

    print(f'\n  Feature breakdown:')
    print(f'    FE_ engineered : {fe_count} (of which D_normalized: {d_norm})')
    print(f'    _isnan flags   : {nan_count}')
    print(f'    Raw features   : {raw_count}')

    log_check('Engineered features in final set', 'PASS' if fe_count > 0 else 'WARN',
              f'{fe_count} FE_ features ({d_norm} D_normalized)')
    log_check('NaN flag features in final set', 'PASS' if nan_count > 0 else 'WARN',
              f'{nan_count} _isnan features')

    for split in ['train', 'val', 'test']:
        fpath = FEATURES_DIR / f'{split}_fraud_features.parquet'
        if fpath.exists():
            df_tmp = pd.read_parquet(fpath)
            log_check(f'{split}_fraud_features.parquet', 'PASS', f'Shape: {df_tmp.shape}')
        else:
            log_check(f'{split}_fraud_features.parquet', 'FAIL', f'File not found: {fpath}')

    train_f = pd.read_parquet(FEATURES_DIR / 'train_fraud_features.parquet')
    val_f   = pd.read_parquet(FEATURES_DIR / 'val_fraud_features.parquet')
    diff    = set(train_f.columns).symmetric_difference(set(val_f.columns))
    if not diff:
        log_check('Train/Val feature alignment', 'PASS', f'Both have {len(train_f.columns)} identical columns')
    else:
        log_check('Train/Val feature alignment', 'WARN', f'{len(diff)} columns differ: {list(diff)[:5]}')

---
## Check 5 — Target Leakage

No feature should use future information or be derived from the target variable `isFraud`. Verifying that `PEAK_FRAUD_HOURS` is a pre-computed constant (not computed inside FE using labels) and that `TransactionDT` is excluded from the final feature set.

In [ ]:
print('=' * 60)
print('CHECK 5: TARGET LEAKAGE')
print('=' * 60)

fe_path_py = ROOT_DIR / 'src' / 'features' / 'ieee_cis' / 'feature_engineer.py'
with open(fe_path_py, encoding='utf-8') as f:
    fe_source = f.read()

print('\n  Checking PEAK_FRAUD_HOURS derivation...')
if 'PEAK_FRAUD_HOURS = [5, 6, 7, 8, 9]' in fe_source:
    log_check('PEAK_FRAUD_HOURS — pre-computed constant', 'PASS',
              'Hardcoded from EDA — not computed inside FE function using target')
else:
    log_check('PEAK_FRAUD_HOURS derivation', 'WARN', 'Cannot verify PEAK_FRAUD_HOURS = [5,6,7,8,9]')

print('\n  Checking for isFraud usage in feature_engineer.py...')
fe_lines   = fe_source.split('\n')
suspicious = [
    (i+1, line.strip()) for i, line in enumerate(fe_lines)
    if 'isFraud' in line and not line.strip().startswith('#')
]
if not suspicious:
    log_check('isFraud not used in feature_engineer.py', 'PASS',
              'No target variable usage in feature engineering')
else:
    log_check('isFraud usage in feature_engineer.py', 'FAIL',
              f'Found at lines: {[s[0] for s in suspicious]}')
    for lineno, line in suspicious:
        print(f'    Line {lineno}: {line}')

print('\n  Checking TransactionDT in final features...')
train_f = pd.read_parquet(FEATURES_DIR / 'train_fraud_features.parquet')
if 'TransactionDT' not in train_f.columns:
    log_check('TransactionDT excluded from features', 'PASS',
              'Raw timestamp not in feature set — only FE_hour, FE_dayofweek etc.')
else:
    log_check('TransactionDT in feature set', 'WARN',
              'Raw TransactionDT is in features — may cause leakage via timestamp')

print('\n  Verifying FE_hour distribution...')
if 'FE_hour' in train_f.columns:
    print(f'  FE_hour range : {train_f["FE_hour"].min()} -> {train_f["FE_hour"].max()}')
    if train_f['FE_hour'].min() >= 0 and train_f['FE_hour'].max() <= 23:
        log_check('FE_hour range valid (0-23)', 'PASS',
                  f'Hours: {train_f["FE_hour"].min()} -> {train_f["FE_hour"].max()}')
    else:
        log_check('FE_hour range', 'FAIL', f'Invalid hour values detected')
else:
    log_check('FE_hour in features', 'WARN', 'FE_hour not found in final feature set')

---
## Check 6 — Stacking CV Leakage

Stacking ensemble must use **TimeSeriesSplit**, not StratifiedKFold. Random shuffle on temporal data creates future-to-past leakage in OOF predictions.

In [ ]:
print('=' * 60)
print('CHECK 6: STACKING CV LEAKAGE')
print('=' * 60)

detector_path = ROOT_DIR / 'src' / 'models' / 'fraud' / 'fraud_detector.py'

if not detector_path.exists():
    log_check('fraud_detector.py exists', 'WARN', f'Not found at {detector_path}')
else:
    with open(detector_path, encoding='utf-8') as f:
        detector_source = f.read()

    stacking_start = detector_source.find('def train_stacking')
    stacking_section = detector_source[stacking_start:stacking_start + 3000] if stacking_start >= 0 else ''

    if 'StratifiedKFold' in stacking_section:
        log_check('Stacking CV — no StratifiedKFold', 'FAIL',
                  'StratifiedKFold in train_stacking() — temporal leakage!')
    else:
        log_check('Stacking CV — no StratifiedKFold', 'PASS',
                  'StratifiedKFold not found in train_stacking()')

    if 'TimeSeriesSplit' in stacking_section:
        log_check('Stacking CV — TimeSeriesSplit', 'PASS',
                  'Temporal-aware cross-validation used')
    else:
        log_check('Stacking CV — TimeSeriesSplit missing', 'FAIL',
                  'TimeSeriesSplit not found in train_stacking()')

    if 'LogisticRegression' in stacking_section:
        log_check('Stacking meta-learner — LogisticRegression', 'PASS',
                  'Linear meta-learner — less prone to overfitting than XGB')
    else:
        log_check('Stacking meta-learner', 'WARN', 'LogisticRegression not found in train_stacking()')

    if 'CalibratedClassifierCV' in stacking_section:
        log_check('Stacking — calibrated base models', 'PASS',
                  'isotonic calibration applied per fold before meta-learner')
    else:
        log_check('Stacking calibration', 'WARN', 'CalibratedClassifierCV not found')

    import re
    spw = re.findall(r'scale_pos_weight["\s]*[:=]\s*(\d+)', detector_source)
    if '28' in spw or '27' in spw:
        log_check('scale_pos_weight = 28 (EDA: ratio=27.6)', 'PASS', f'Found values: {spw}')
    else:
        log_check('scale_pos_weight value', 'WARN', f'Found: {spw} — expected 28')

---
## Final Summary — All Checks

> **Saved:** `outputs/figures/fraud/eda/leakage/leakage_summary.png`

In [ ]:
print('\n' + '=' * 60)
print('LEAKAGE ANALYSIS — FINAL SUMMARY')
print('=' * 60)

df_results = pd.DataFrame(results)
n_pass = (df_results['status'] == 'PASS').sum()
n_warn = (df_results['status'] == 'WARN').sum()
n_fail = (df_results['status'] == 'FAIL').sum()
total  = len(df_results)

print(f'\n  Total checks : {total}')
print(f'  PASS         : {n_pass}  ({n_pass/total*100:.0f}%)')
print(f'  WARN         : {n_warn}  ({n_warn/total*100:.0f}%)')
print(f'  FAIL         : {n_fail}  ({n_fail/total*100:.0f}%)')

print('\n' + '-' * 60)
if n_fail == 0 and n_warn == 0:
    print('  RESULT: ALL CLEAR — Pipeline ready for training')
elif n_fail == 0:
    print('  RESULT: WARNINGS PRESENT — Review before training')
else:
    print('  RESULT: FAILURES DETECTED — Fix before training')
print('-' * 60)

print('\nFAIL items:')
fails = df_results[df_results['status'] == 'FAIL']
if len(fails) == 0:
    print('  None')
else:
    for _, row in fails.iterrows():
        print(f'  [FAIL] {row["check"]}')
        print(f'         {row["detail"]}')

print('\nWARN items:')
warns = df_results[df_results['status'] == 'WARN']
if len(warns) == 0:
    print('  None')
else:
    for _, row in warns.iterrows():
        print(f'  [WARN] {row["check"]}')
        print(f'         {row["detail"]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pie_vals    = [n_pass, n_warn, n_fail]
pie_labels  = [f'PASS\n{n_pass}', f'WARN\n{n_warn}', f'FAIL\n{n_fail}']
pie_colors  = [PALETTE['pass'], PALETTE['warn'], PALETTE['fail']]
pie_explode = [0.03, 0.03, 0.08 if n_fail > 0 else 0.03]
axes[0].pie(pie_vals, labels=pie_labels, colors=pie_colors,
            autopct='%1.0f%%', startangle=90, explode=pie_explode,
            textprops={'fontsize': 12})
axes[0].set_title('Check Results Distribution', fontweight='bold')

status_colors = {'PASS': PALETTE['pass'], 'WARN': PALETTE['warn'], 'FAIL': PALETTE['fail']}
bar_colors    = [status_colors.get(s, 'gray') for s in df_results['status']]
axes[1].barh(range(len(df_results)), [1]*len(df_results), color=bar_colors, edgecolor='black')
axes[1].set_yticks(range(len(df_results)))
axes[1].set_yticklabels(
    [f"{row['status']}: {row['check'][:50]}" for _, row in df_results.iterrows()],
    fontsize=8
)
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1.5)
axes[1].set_xticks([])
axes[1].set_title('All Checks — Status', fontweight='bold')

patches = [mpatches.Patch(color=PALETTE['pass'], label='PASS'),
           mpatches.Patch(color=PALETTE['warn'], label='WARN'),
           mpatches.Patch(color=PALETTE['fail'], label='FAIL')]
axes[1].legend(handles=patches, loc='lower right')

plt.suptitle(f'Leakage Analysis Summary — PASS:{n_pass} | WARN:{n_warn} | FAIL:{n_fail}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'leakage_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: leakage/leakage_summary.png')